# Solana Sniper Bot: leak-free public reproduction

This notebook reproduces the corrected development-only evidence in the Kaggle Writeup.

**Decision boundary:** `t_decision` is the token deployment transaction. Classifier and entry-selection features must exist when that transaction is observed. Strict signer history uses only smaller block slots. Post-deployment trades, candles, prices, realized PnL, outcome labels, and future signer activity are forbidden as features; post-deployment trades appear only in the backtest cells.

The notebook contains code and aggregate outputs, not competition rows. Authorized participants must obtain and privately stage the official files as documented in `notebooks/README.md`. The final chronological holdout remains sealed throughout this notebook.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists(), "Run from the repository root"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ.setdefault("SOLANA_SNIPER_DATA_ROOT", str(PROJECT_ROOT / "data"))

from solana_sniper.manifest import sha256_file  # noqa: E402
from solana_sniper.paths import PROCESSED_DIR, RAW_DIR, REPORT_DIR  # noqa: E402

EXPECTED_CLASSIFICATION_SHA256 = "57af874b0768eaf43c54f04d698cda9e3c3e1d9bf3e1a25c7c69910ecbe8817f"
CLASSIFICATION_PATH = PROCESSED_DIR / "classification_dataset_creator_history.parquet"
HISTORY_MANIFEST_PATH = PROCESSED_DIR / "creator_history_manifest.json"
print({"project_root_detected": PROJECT_ROOT.name, "private_data_root_configured": True})

{'project_root_detected': 'solana-sniper-bot', 'private_data_root_configured': True}


## 1. Source, schema, and `t_decision` audit

This cell loads only deployment-time modeling tables. It verifies hashes, uniqueness, canonical UTC fields, and the strict-history manifest before any model is fit.

In [2]:
required_paths = [
    CLASSIFICATION_PATH,
    HISTORY_MANIFEST_PATH,
    PROCESSED_DIR / "entry_latency.parquet",
    PROCESSED_DIR / "replica_entry_prices.parquet",
    RAW_DIR / "wallet" / "5brv79e_activity.parquet",
    RAW_DIR / "june" / "pumpfun_trades.parquet",
    REPORT_DIR / "entry_prices_metrics.json",
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f"Missing privately staged official artifacts: {missing}"

classification_sha256 = sha256_file(CLASSIFICATION_PATH)
assert classification_sha256 == EXPECTED_CLASSIFICATION_SHA256
frame = pd.read_parquet(CLASSIFICATION_PATH)
history_manifest = json.loads(HISTORY_MANIFEST_PATH.read_text(encoding="utf-8"))
canonical_time = pd.to_datetime(frame["decision_time"], utc=True)
audit = {
    "rows": len(frame),
    "unique_tokens": frame["token_address"].nunique(),
    "positive_rows": int(frame["label"].eq(1).sum()),
    "negative_rows": int(frame["label"].eq(0).sum()),
    "duplicate_tokens": int(frame["token_address"].duplicated().sum()),
    "utc_hour_mismatches": int(frame["decision_hour_utc"].ne(canonical_time.dt.hour).sum()),
    "utc_weekday_mismatches": int(
        frame["decision_weekday_utc"].ne(canonical_time.dt.dayofweek).sum()
    ),
    "strict_history_violations": int(history_manifest["strict_time_violations"]),
    "classification_sha256": classification_sha256,
}
assert audit["rows"] == audit["unique_tokens"] == history_manifest["rows"]
assert history_manifest["sha256"] == classification_sha256
assert (
    audit["duplicate_tokens"]
    == audit["utc_hour_mismatches"]
    == audit["utc_weekday_mismatches"]
    == audit["strict_history_violations"]
    == 0
)
pd.Series(audit, name="value").to_frame()

,value
rows,218350
unique_tokens,218350
positive_rows,15927
negative_rows,202423
duplicate_tokens,0
utc_hour_mismatches,0
utc_weekday_mismatches,0
strict_history_violations,0
classification_sha256,57af874b0768eaf43c54f04d698cda9e3c3e1d9bf3e1a2...


## 2. Chronological model stability

The model is evaluated with population weights and expanding time folds. PR-AUC, precision, recall, and F1—not accuracy or ROC-AUC—are the decision metrics. The final holdout is not predicted.

In [3]:
from solana_sniper.creator_history_stability import run_stability

stability = run_stability(CLASSIFICATION_PATH)
assert stability["dataset_sha256"] == EXPECTED_CLASSIFICATION_SHA256
assert stability["final_holdout_evaluated"] is False
stability_rows = []
for fold in stability["expanding_folds"]:
    stability_rows.append(
        {
            "fold": fold["fold"],
            "without_history_pr_auc": fold["without_creator_history"]["population_adjusted_pr_auc"],
            "with_history_pr_auc": fold["with_creator_history"]["population_adjusted_pr_auc"],
            "delta": fold["pr_auc_delta"],
            **fold["with_creator_history"]["selected_operating_point"],
        }
    )
pd.DataFrame(stability_rows)

,fold,without_history_pr_auc,with_history_pr_auc,delta,threshold,precision,recall,f1
0,1,0.030783,0.063581,0.032799,0.895596,0.110299,0.196639,0.141326
1,2,0.057802,0.081463,0.023660,0.910429,0.133512,0.202069,0.160788
2,3,0.055216,0.073577,0.018361,0.927085,0.137931,0.211041,0.166828


## 3. Temporal feature attribution

Permutation occurs only inside development validation folds. It measures association, not causality, and correlated feature drops are not additive.

In [4]:
from solana_sniper.feature_attribution import run_feature_attribution

attribution = run_feature_attribution(CLASSIFICATION_PATH)
assert attribution["dataset_sha256"] == EXPECTED_CLASSIFICATION_SHA256
assert attribution["final_holdout_evaluated"] is False
group_rows = sorted(attribution["group_importance"], key=lambda row: row["rank"])
pd.DataFrame(group_rows)[
    ["name", "mean_temporal_pr_auc_drop", "minimum_temporal_pr_auc_drop", "positive_fold_count"]
]

,name,mean_temporal_pr_auc_drop,minimum_temporal_pr_auc_drop,positive_fold_count
0,creator_history,0.056611,0.049283,3
1,transaction_structure,0.052615,0.046286,3
2,metadata,0.011476,0.005699,3
3,fee_and_compute,0.008584,0.006367,3
4,transaction_error,0.000000,0.000000,0
5,decision_time,-0.000285,-0.002911,2


## 4. Requested 0/1/2-slot replica backtest

Post-deployment trades are introduced here for the first time. They are used only to mark observed entries and exits. Requested delay is reported separately from actual fill delay.

In [5]:
from solana_sniper.replica_backtest import run_validation_backtest

replica = run_validation_backtest()
assert replica["classification_dataset_sha256"] == EXPECTED_CLASSIFICATION_SHA256
assert replica["final_holdout_evaluated"] is False
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "replica_validation_backtest.json").write_text(
    json.dumps(replica, indent=2) + "\n", encoding="utf-8", newline="\n"
)
replica_rows = [
    row
    for row in replica["backtest_results"]
    if row["execution_policy"] == "all_observed_proxy"
    and row["fee_scenario"] == "training_median_fee"
]
pd.DataFrame(replica_rows)[
    [
        "requested_delay_slots",
        "coverage_rate_population_weighted",
        "actual_delay_slots",
        "net_mean_return",
        "net_median_return_unweighted",
        "net_hit_rate",
        "max_drawdown_fraction",
        "insolvent_under_capital_model",
    ]
]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,requested_delay_slots,coverage_rate_population_weighted,actual_delay_slots,net_mean_return,net_median_return_unweighted,net_hit_rate,max_drawdown_fraction,insolvent_under_capital_model
0,0,1.0,"{'median': 0.0, 'p90': 1.0, 'max': 361}",0.748598,0.160836,0.494274,0.032032,False
1,1,1.0,"{'median': 1.0, 'p90': 2.0, 'max': 361}",-0.043152,-0.102542,0.309208,2.330803,True
2,2,1.0,"{'median': 2.0, 'p90': 4.0, 'max': 361}",0.328586,-0.084739,0.281264,0.082936,False


## 5. Transaction-position feasibility

The optimistic zero-slot entry is replaced with the first same-slot trade no earlier than the target wallet's training-only median transaction-position gap. This is still a position proxy, not proof of live reaction.

In [6]:
from solana_sniper.position_backtest import run_position_backtest

position = run_position_backtest()
assert position["classification_dataset_sha256"] == EXPECTED_CLASSIFICATION_SHA256
assert position["final_holdout_evaluated"] is False
(REPORT_DIR / "position_lag_validation_backtest.json").write_text(
    json.dumps(position, indent=2) + "\n", encoding="utf-8", newline="\n"
)
pd.DataFrame(position["backtest_results"])[
    [
        "fee_scenario",
        "executed_sample_rows",
        "executed_population_weight",
        "net_mean_return",
        "net_median_return_unweighted",
        "net_hit_rate",
        "max_drawdown_fraction",
        "insolvent_under_capital_model",
    ]
]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,fee_scenario,executed_sample_rows,executed_population_weight,net_mean_return,net_median_return_unweighted,net_hit_rate,max_drawdown_fraction,insolvent_under_capital_model
0,gross,237,1245.0,0.154111,0.017447,0.504418,0.358815,False
1,training_median_fee,237,1245.0,0.081088,-0.055575,0.435341,0.919625,True
2,training_p90_fee,237,1245.0,-0.032094,-0.168758,0.356627,3.198987,True


## 6. Target-wallet versus replica comparison

Target-wallet cash flows use actual entries and sizing. Replica results use fixed notional and population weighting, so total dollars are not directly comparable.

In [7]:
from solana_sniper.competitor_pnl import run_competitor_pnl

competitor = run_competitor_pnl()
assert competitor["classification_dataset_sha256"] == EXPECTED_CLASSIFICATION_SHA256
assert competitor["classifier_final_holdout_evaluated"] is False
assert competitor["replica_final_holdout_evaluated"] is False
comparison = competitor["development_head_to_head"]
pd.DataFrame(
    [comparison["target_wallet"], comparison["position_lag_replica"]],
    index=["target_wallet", "position_lag_replica"],
)

,entry_set,token_rows,net_mean_return,net_median_return,hit_rate,max_drawdown_fraction,net_pnl_usd,executed_sample_rows,executed_population_weight
target_wallet,actual_target_wallet_buys,1372.0,0.107118,0.030496,0.566327,0.068893,48847.577700,NaN,NaN
position_lag_replica,model_selected_population_weighted_candidates,NaN,0.081088,-0.055575,0.435341,0.919625,20308.163975,237.0,1245.0


## 7. Final integrity certificate

The candidate remains a development-only result. The profitable first-trade zero-slot proxy is an optimistic upper bound; the position-lag candidate is rejected because its typical trade loses and its tight-capital path crosses zero.

In [8]:
holdout_start = pd.Timestamp(replica["final_holdout_start_utc"])
assert pd.Timestamp(replica["max_backtest_outcome_utc"]) < holdout_start
assert pd.Timestamp(position["max_backtest_outcome_utc"]) < holdout_start
assert position["acceptance"]["all_checks_passed"] is False
certificate = {
    "classification_sha256": EXPECTED_CLASSIFICATION_SHA256,
    "strict_history_violations": audit["strict_history_violations"],
    "utc_clock_mismatches": audit["utc_hour_mismatches"] + audit["utc_weekday_mismatches"],
    "standard_validation_pr_auc": stability["standard_validation"]["with_creator_history"][
        "population_adjusted_pr_auc"
    ],
    "operating_point": stability["standard_validation"]["with_creator_history"][
        "selected_operating_point"
    ],
    "replica_viable_delays": replica["acceptance"]["observed_viable_delays"],
    "position_lag_decision": position["acceptance"]["decision"],
    "target_development_mean_roi": competitor["development_target_wallet"]["net_mean_roi"],
    "target_development_median_roi": competitor["development_target_wallet"]["net_median_roi"],
    "final_holdout_evaluated": False,
}
certificate

{'classification_sha256': '57af874b0768eaf43c54f04d698cda9e3c3e1d9bf3e1a25c7c69910ecbe8817f',
 'strict_history_violations': 0,
 'utc_clock_mismatches': 0,
 'standard_validation_pr_auc': 0.07028141318814668,
 'operating_point': {'threshold': 0.9370987253131746,
  'precision': 0.1203377902885292,
  'recall': 0.2011173184357542,
  'f1': 0.15057787561915245},
 'replica_viable_delays': [0],
 'position_lag_decision': 'rejected_position_lag_fails_fee_robust_capital_criterion',
 'target_development_mean_roi': 0.107118407620444,
 'target_development_median_roi': 0.030496071271284744,
 'final_holdout_evaluated': False}